<a href="https://colab.research.google.com/github/kamalrajarulprakasam-sudo/GENAI-GOLD-Badge-Assignments/blob/main/Problem3_GPT4All_Gemini_RAG_ChromaDB/Problem3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem 3 — GPT4All + Gemini Dual RAG with ChromaDB (Colab)

Self-contained Colab version of `Problem3_GPT4All_Gemini_RAG_ChromaDB`. It:

1. Chunks + embeds sample documents into a persistent **ChromaDB** collection.
2. Answers each question with **two** LLMs using the same retrieved context:
   - a local, offline **GPT4All** model (auto-downloaded, ~2GB the first time), and
   - the cloud **Gemini** API (needs a free API key from https://aistudio.google.com/apikey).
3. Optionally asks Gemini to reconcile both answers into one final response.


## 1. Install Python dependencies

In [1]:
!pip install -q chromadb pypdf gpt4all google-generativeai streamlit python-dotenv


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95

## 2. Set your Gemini API key

Leave the prompt blank to skip Gemini and only use the local GPT4All model.

In [2]:
import os
from getpass import getpass

if not os.environ.get('GEMINI_API_KEY'):
    key = getpass('Enter your Gemini API key (or leave blank to skip Gemini): ')
    if key:
        os.environ['GEMINI_API_KEY'] = key


Enter your Gemini API key (or leave blank to skip Gemini): ··········


## 3. Write out the project source files

In [3]:
%%writefile rag_pipeline.py
"""Core RAG pipeline: document loading, chunking, ChromaDB store, and dual
generation with the local **GPT4All** desktop LLM and the cloud **Gemini**
API.

Shared by `ingest.py`, `app.py` (Streamlit UI) and `cli.py`.

Design
------
1. Documents (`.txt` and `.pdf`) under `documents/` are chunked and embedded
   into a persistent ChromaDB collection (using Chroma's built-in default
   embedding function, so no extra embedding model/API is required).
2. A query is embedded and the top-k most similar chunks are retrieved from
   Chroma -- this part is identical regardless of which LLM answers.
3. The same retrieved context is sent to *two* generators:
     - `answer_with_gpt4all`: a local GPT4All model (fully offline).
     - `answer_with_gemini`:  Google's Gemini API (needs GEMINI_API_KEY).
   `answer_dual` runs both and optionally asks Gemini to reconcile the two
   answers into one final, cited response.
"""

from __future__ import annotations

import glob
import os
from dataclasses import dataclass
from pathlib import Path

import chromadb
from pypdf import PdfReader

try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:  # python-dotenv is optional at import time
    pass

BASE_DIR = Path(__file__).parent
DOCS_DIR = BASE_DIR / "documents"
CHROMA_DIR = BASE_DIR / "chroma_db"
COLLECTION_NAME = "gpt4all_gemini_rag_collection"

# Local GPT4All model. Auto-downloaded on first use into ~/.cache/gpt4all.
# Swap for any model name from https://gpt4all.io/models/models3.json
GPT4ALL_MODEL_NAME = os.environ.get("GPT4ALL_MODEL_NAME", "orca-mini-3b-gguf2-q4_0.gguf")

# Gemini model used for the cloud answer + optional reconciliation step.
GEMINI_MODEL_NAME = os.environ.get("GEMINI_MODEL_NAME", "gemini-1.5-flash")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

CHUNK_SIZE = 1200        # characters per chunk
CHUNK_OVERLAP = 200      # characters shared between consecutive chunks
TOP_K = 4                # number of chunks retrieved per query


@dataclass
class RetrievedChunk:
    text: str
    source: str
    chunk_index: int
    distance: float


@dataclass
class DualAnswer:
    query: str
    chunks: list[RetrievedChunk]
    gpt4all_answer: str
    gemini_answer: str
    synthesized_answer: str | None = None


# --------------------------------------------------------------------------
# Chunking + ingestion
# --------------------------------------------------------------------------

def _chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Simple sliding-window character chunker with overlap."""
    text = " ".join(text.split())  # normalize whitespace
    if not text:
        return []

    chunks: list[str] = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks


def extract_text(doc_path: Path) -> str:
    if doc_path.suffix.lower() == ".pdf":
        reader = PdfReader(str(doc_path))
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    return doc_path.read_text(encoding="utf-8", errors="ignore")


def get_chroma_collection():
    """Return (creating if necessary) the persistent Chroma collection.

    Uses Chroma's default embedding function (ONNX MiniLM) so retrieval
    works even without GPT4All or a Gemini API key configured.
    """
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    return client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )


def ingest_documents(docs_dir: Path = DOCS_DIR, reset: bool = True) -> int:
    """Chunk every .txt/.pdf under `docs_dir` and (re)populate ChromaDB.

    Returns the number of chunks indexed.
    """
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    if reset:
        try:
            client.delete_collection(COLLECTION_NAME)
        except Exception:
            pass
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )

    doc_paths = sorted(
        glob.glob(str(docs_dir / "*.txt")) + glob.glob(str(docs_dir / "*.pdf"))
    )
    if not doc_paths:
        raise FileNotFoundError(
            f"No .txt or .pdf files found in {docs_dir}. Run create_sample_docs.py "
            "first, or drop your own documents into that folder."
        )

    ids, documents, metadatas = [], [], []
    for doc_path in doc_paths:
        source_name = Path(doc_path).name
        text = extract_text(Path(doc_path))
        chunks = _chunk_text(text)
        for i, chunk in enumerate(chunks):
            ids.append(f"{source_name}::{i}")
            documents.append(chunk)
            metadatas.append({"source": source_name, "chunk_index": i})

    if documents:
        # Chroma has a per-call batch size limit; chunk the insert.
        batch_size = 128
        for start in range(0, len(documents), batch_size):
            end = start + batch_size
            collection.add(
                ids=ids[start:end],
                documents=documents[start:end],
                metadatas=metadatas[start:end],
            )

    return len(documents)


def retrieve(query: str, top_k: int = TOP_K) -> list[RetrievedChunk]:
    collection = get_chroma_collection()
    results = collection.query(query_texts=[query], n_results=top_k)

    retrieved: list[RetrievedChunk] = []
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    dists = results.get("distances", [[]])[0]
    for doc, meta, dist in zip(docs, metas, dists):
        retrieved.append(
            RetrievedChunk(
                text=doc,
                source=meta.get("source", "unknown"),
                chunk_index=meta.get("chunk_index", -1),
                distance=dist,
            )
        )
    return retrieved


SYSTEM_PROMPT = (
    "You are a helpful assistant answering questions using ONLY the provided "
    "context excerpts from the user's documents. If the answer is not "
    "contained in the context, say you don't know instead of guessing. "
    "Always mention which source document(s) you used."
)


def build_prompt(query: str, chunks: list[RetrievedChunk]) -> str:
    context_blocks = []
    for c in chunks:
        context_blocks.append(f"[Source: {c.source}, chunk {c.chunk_index}]\n{c.text}")
    context = "\n\n---\n\n".join(context_blocks) if context_blocks else "(no context retrieved)"

    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer the question using only the context above."
    )


# --------------------------------------------------------------------------
# Generator 1: local GPT4All desktop LLM
# --------------------------------------------------------------------------

_gpt4all_model = None  # lazy-loaded singleton, avoids reloading weights per call


def _get_gpt4all_model():
    global _gpt4all_model
    if _gpt4all_model is None:
        from gpt4all import GPT4All

        _gpt4all_model = GPT4All(GPT4ALL_MODEL_NAME, allow_download=True)
    return _gpt4all_model


def answer_with_gpt4all(query: str, chunks: list[RetrievedChunk] | None = None, top_k: int = TOP_K) -> tuple[str, list[RetrievedChunk]]:
    """RAG turn answered by the local GPT4All model."""
    if chunks is None:
        chunks = retrieve(query, top_k=top_k)
    prompt = build_prompt(query, chunks)

    model = _get_gpt4all_model()
    with model.chat_session():
        answer = model.generate(prompt, max_tokens=512, temp=0.2)
    return answer.strip(), chunks


# --------------------------------------------------------------------------
# Generator 2: Gemini API
# --------------------------------------------------------------------------

_gemini_model = None


def _get_gemini_model():
    global _gemini_model
    if _gemini_model is None:
        if not GEMINI_API_KEY:
            raise RuntimeError(
                "GEMINI_API_KEY is not set. Get a key from https://aistudio.google.com/apikey "
                "and set it as an environment variable (or in a local .env file)."
            )
        import google.generativeai as genai

        genai.configure(api_key=GEMINI_API_KEY)
        _gemini_model = genai.GenerativeModel(GEMINI_MODEL_NAME)
    return _gemini_model


def answer_with_gemini(query: str, chunks: list[RetrievedChunk] | None = None, top_k: int = TOP_K) -> tuple[str, list[RetrievedChunk]]:
    """RAG turn answered by the Gemini API."""
    if chunks is None:
        chunks = retrieve(query, top_k=top_k)
    prompt = build_prompt(query, chunks)

    model = _get_gemini_model()
    response = model.generate_content(prompt)
    return response.text.strip(), chunks


# --------------------------------------------------------------------------
# Combined: retrieve once, ask both LLMs, optionally reconcile with Gemini
# --------------------------------------------------------------------------

def answer_dual(query: str, top_k: int = TOP_K, synthesize: bool = True) -> DualAnswer:
    """Retrieve context once and get answers from both GPT4All and Gemini.

    If `synthesize` is True (and Gemini is configured), Gemini is asked a
    second time to reconcile the two answers into a single best response
    grounded in the same retrieved context.
    """
    chunks = retrieve(query, top_k=top_k)

    try:
        gpt4all_answer, _ = answer_with_gpt4all(query, chunks=chunks)
    except Exception as exc:  # model not installed / download failed / etc.
        gpt4all_answer = f"[GPT4All error] {exc}"

    try:
        gemini_answer, _ = answer_with_gemini(query, chunks=chunks)
    except Exception as exc:  # missing API key / network error / etc.
        gemini_answer = f"[Gemini error] {exc}"

    synthesized_answer = None
    if synthesize and GEMINI_API_KEY:
        try:
            synthesized_answer = _reconcile_with_gemini(query, chunks, gpt4all_answer, gemini_answer)
        except Exception as exc:
            synthesized_answer = f"[Synthesis error] {exc}"

    return DualAnswer(
        query=query,
        chunks=chunks,
        gpt4all_answer=gpt4all_answer,
        gemini_answer=gemini_answer,
        synthesized_answer=synthesized_answer,
    )


def _reconcile_with_gemini(
    query: str,
    chunks: list[RetrievedChunk],
    gpt4all_answer: str,
    gemini_answer: str,
) -> str:
    """Ask Gemini to act as a judge/reconciler over the two candidate answers."""
    context_blocks = "\n\n---\n\n".join(f"[Source: {c.source}]\n{c.text}" for c in chunks)
    prompt = (
        "You are reviewing two candidate answers to the same question, both "
        "grounded in the same document context. Produce one final answer that "
        "keeps whatever is correct and well-supported, corrects any "
        "hallucinated or unsupported claims, and cites the source document(s). "
        "If both answers already agree, just clean up the wording.\n\n"
        f"Context:\n{context_blocks}\n\n"
        f"Question: {query}\n\n"
        f"Candidate answer A (GPT4All, local model):\n{gpt4all_answer}\n\n"
        f"Candidate answer B (Gemini):\n{gemini_answer}\n\n"
        "Final reconciled answer:"
    )
    model = _get_gemini_model()
    response = model.generate_content(prompt)
    return response.text.strip()


Writing rag_pipeline.py


In [4]:
%%writefile create_sample_docs.py
"""Generate 3 sample text documents used to demonstrate the RAG pipeline.

Run once before `ingest.py`:

    python create_sample_docs.py

This creates `documents/renewable_energy.txt`, `documents/space_exploration.txt`
and `documents/personal_finance.txt`. Feel free to replace these with your own
`.txt` or `.pdf` files -- the ingestion script picks up every `*.txt` and
`*.pdf` file it finds in the `documents/` folder.
"""

from __future__ import annotations

import textwrap
from pathlib import Path

DOCS_DIR = Path(__file__).parent / "documents"

DOCUMENTS: dict[str, str] = {
    "renewable_energy.txt": """
        Renewable Energy Basics

        Solar Power: Photovoltaic (PV) panels convert sunlight directly into
        electricity using semiconductor cells. Utility-scale solar farms now
        produce electricity at a lower levelized cost than most fossil-fuel
        plants in sunny regions. Residential rooftop systems typically pay
        back their installation cost within 6 to 10 years depending on local
        electricity rates and incentives.

        Wind Power: Modern wind turbines convert kinetic energy from moving
        air into electricity via a rotor, gearbox, and generator. Offshore
        wind farms benefit from stronger, steadier winds than onshore sites
        but cost more to install and maintain due to marine foundations and
        subsea cabling.

        Energy Storage: Because solar and wind output varies with weather
        and time of day, grid-scale battery storage (mostly lithium-ion) is
        increasingly paired with renewables to shift generation to periods
        of high demand. Pumped-hydro storage remains the largest form of
        grid storage worldwide by capacity.

        Grid Integration: Utilities use demand forecasting, battery
        storage, and flexible natural-gas peaker plants to balance the
        intermittency of renewables. Smart grids allow two-way
        communication between utilities and consumers to shift load to
        times of abundant renewable supply.

        Policy Incentives: Many countries offer tax credits, feed-in
        tariffs, or renewable portfolio standards to accelerate adoption.
        The U.S. federal solar Investment Tax Credit (ITC), for example,
        allows homeowners and businesses to deduct a percentage of solar
        installation costs from their taxes.
        """,
    "space_exploration.txt": """
        A Brief Timeline of Space Exploration

        1957 - Sputnik 1: The Soviet Union launches the first artificial
        satellite, kicking off the Space Race with the United States.

        1969 - Apollo 11: NASA astronauts Neil Armstrong and Buzz Aldrin
        become the first humans to walk on the Moon, while Michael Collins
        orbits above in the command module.

        1981 - Space Shuttle: NASA's Space Shuttle Columbia becomes the
        first reusable crewed spacecraft, flying dozens of missions over
        three decades to deploy satellites and build the International
        Space Station.

        1998 - International Space Station (ISS): Construction begins on
        the ISS, a multi-nation collaboration that has hosted continuous
        human presence in low Earth orbit since November 2000.

        2012 - Commercial Cargo: SpaceX's Dragon capsule becomes the first
        commercial spacecraft to dock with the ISS, marking the start of
        commercial resupply missions.

        2020 - Crewed Commercial Flight: SpaceX's Crew Dragon carries NASA
        astronauts to the ISS, the first crewed orbital flight launched
        from U.S. soil since the Shuttle retired in 2011.

        2021-Present - Artemis Program: NASA's Artemis program aims to
        return humans to the lunar surface, including the first woman and
        person of color, and to establish a sustainable lunar presence as
        a stepping stone toward crewed Mars missions.
        """,
    "personal_finance.txt": """
        Personal Finance 101

        Budgeting: The 50/30/20 rule suggests allocating 50% of after-tax
        income to needs (housing, groceries, utilities), 30% to wants
        (dining out, entertainment), and 20% to savings and debt
        repayment. Tracking expenses for a month is the first step to
        building an accurate budget.

        Emergency Fund: Financial advisors typically recommend keeping 3
        to 6 months of essential living expenses in a readily accessible
        savings account before investing aggressively, to avoid going into
        debt when unexpected expenses arise.

        Compound Interest: Money invested early benefits from compound
        growth, where returns are reinvested and themselves earn returns.
        Starting to invest in your 20s instead of your 30s can roughly
        double the final retirement balance for the same monthly
        contribution, due to the extra decade of compounding.

        Debt Management: The "avalanche" method pays off debts with the
        highest interest rate first while making minimum payments on
        others, minimizing total interest paid. The "snowball" method pays
        off the smallest balance first for psychological motivation, even
        though it usually costs slightly more in total interest.

        Retirement Accounts: Employer-sponsored plans (like a 401(k) in
        the U.S.) often include matching contributions -- effectively free
        money -- so contributing at least enough to get the full match is
        usually recommended before investing elsewhere. Tax-advantaged
        individual accounts (IRAs) offer additional tax-deferred or
        tax-free growth depending on the account type.
        """,
}


def main() -> None:
    DOCS_DIR.mkdir(parents=True, exist_ok=True)
    for filename, body in DOCUMENTS.items():
        out_path = DOCS_DIR / filename
        out_path.write_text(textwrap.dedent(body).strip() + "\n", encoding="utf-8")
        print(f"Wrote {out_path}")


if __name__ == "__main__":
    main()


Writing create_sample_docs.py


In [5]:
%%writefile ingest.py
"""Ingest all .txt/.pdf files in documents/ into the persistent ChromaDB
collection used by the GPT4All + Gemini RAG pipeline.

Usage:
    python ingest.py            # (re)build the index from scratch
    python ingest.py --no-reset # add to the existing index instead
"""

from __future__ import annotations

import argparse

from rag_pipeline import ingest_documents


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--no-reset",
        action="store_true",
        help="Do not wipe the existing collection before ingesting.",
    )
    args = parser.parse_args()

    n_chunks = ingest_documents(reset=not args.no_reset)
    print(f"Indexed {n_chunks} chunks into ChromaDB.")


if __name__ == "__main__":
    main()


Writing ingest.py


In [6]:
%%writefile cli.py
"""Command-line chat REPL for the GPT4All + Gemini RAG assistant.

Prints the local GPT4All answer, the Gemini answer, and (if GEMINI_API_KEY
is set) a Gemini-reconciled final answer -- all grounded in the same
ChromaDB-retrieved context.

Usage:
    python cli.py
"""

from __future__ import annotations

from rag_pipeline import CHROMA_DIR, GEMINI_API_KEY, GEMINI_MODEL_NAME, GPT4ALL_MODEL_NAME, answer_dual


def main() -> None:
    if not CHROMA_DIR.exists():
        print("No index found yet. Run `python ingest.py` first.")
        return

    print(f"GPT4All + Gemini RAG CLI")
    print(f"  GPT4All model: {GPT4ALL_MODEL_NAME}")
    print(f"  Gemini model:  {GEMINI_MODEL_NAME} ({'configured' if GEMINI_API_KEY else 'NO API KEY SET'})")
    print("Type 'exit' to quit.\n")

    while True:
        try:
            query = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print()
            break

        if not query:
            continue
        if query.lower() in {"exit", "quit"}:
            break

        result = answer_dual(query)

        print(f"\n[GPT4All]: {result.gpt4all_answer}\n")
        print(f"[Gemini]: {result.gemini_answer}\n")
        if result.synthesized_answer:
            print(f"[Synthesized]: {result.synthesized_answer}\n")

        print("Sources:")
        for c in result.chunks:
            print(f"  - {c.source} (chunk {c.chunk_index}, distance={c.distance:.3f})")
        print()


if __name__ == "__main__":
    main()


Writing cli.py


## 4. Generate sample documents (or upload your own .txt/.pdf files)

In [7]:
!python create_sample_docs.py


Wrote /content/documents/renewable_energy.txt
Wrote /content/documents/space_exploration.txt
Wrote /content/documents/personal_finance.txt


## 5. Build the ChromaDB index

In [8]:
!python ingest.py


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100% 79.3M/79.3M [00:03<00:00, 27.4MiB/s]
Indexed 6 chunks into ChromaDB.


## 6. Ask a question (dual RAG: GPT4All + Gemini)

The first call also triggers the one-time GPT4All model download (~2GB).

In [ ]:
import importlib
import rag_pipeline as rp
importlib.reload(rp)

result = rp.answer_dual(
    'What is the recommended emergency fund size?', synthesize=True
)
print('GPT4All answer:\n', result.gpt4all_answer)
print('\nGemini answer:\n', result.gemini_answer)
if result.synthesized_answer:
    print('\nSynthesized answer:\n', result.synthesized_answer)


Downloading: 100%|██████████| 1.98G/1.98G [00:36<00:00, 54.7MiB/s]
Verifying: 100%|██████████| 1.98G/1.98G [00:04<00:00, 459MiB/s]


## 7. (Optional) Interactive chat

Run the cell below for an interactive REPL (type `exit` to stop).

In [ ]:
!python cli.py
